<img src="https://fedesarrollo.org.co/logo-Fede.png" alt="Fedesarrollo" style="float: right; width: 300px; height: auto;">
<img src="https://upload.wikimedia.org/wikipedia/commons/f/f9/Logo_BID_Espa%C3%B1ol.png" alt="Banco Interamericano de Desarrollo" style="float: right; width: 220px; height: auto;">

# Proporción de productos que alcanzan un auge exportador en países comparables

Editor: Juan Diego Heredia Niño - jheredia@fedesarrollo.org.co

Septiembre 2026

### Importar librerías

La siguiente celda marca el inicio de un nuevo bloque de trabajo independiente:

**Librerías utilizadas:**
- `pandas` y `numpy`: para manipulación y análisis de datos estructurados.

In [1]:
import pandas as pd
import numpy as np

### Configuración de rutas

Carga las rutas del proyecto y las opciones globales desde `config.py`, instalado como
paquete editable en la raíz del proyecto.

In [2]:
from config import temp

### Lectura de datos

Carga las participaciones de México, Costa Rica y República Dominicana en las importaciones
estadounidenses por producto y año.

In [3]:
df_paises = pd.read_parquet(temp / 'exportaciones' / 'df_paises.parquet')
df_paises.head()

,ReporterISO3,ProductCode,PartnerISO3,Year,TradeValue in 1000 USD,Total TradeValue in 1000 USD,share
0,MEX,010111,USA,1996,185.69,"11,475.58",0.02
1,MEX,010119,USA,1996,"3,296.32","184,169.66",0.02
2,MEX,010120,USA,1996,15.95,151.90,0.10
3,MEX,010210,USA,1996,109.34,"1,823.58",0.06
4,MEX,010290,USA,1996,"194,208.06","1,738,771.91",0.11


### Universo de productos

Descarta los productos cuya participación declarada excede el total, que corresponden a
inconsistencias de la fuente, y cuenta los productos exportados por cada país de referencia.

In [4]:
list_productos = df_paises.query("share >= 1.5")['ProductCode'].unique().tolist()

In [5]:
productos_CRI = df_paises.query("ReporterISO3 == 'CRI'and ProductCode not in @list_productos")['ProductCode'].nunique()
productos_MEX = df_paises.query("ReporterISO3 == 'MEX'and ProductCode not in @list_productos")['ProductCode'].nunique()
productos_DOM = df_paises.query("ReporterISO3 == 'DOM'and ProductCode not in @list_productos")['ProductCode'].nunique()

print(productos_CRI, productos_MEX, productos_DOM)

3411 4565 3452


### Continuidad de las series

Define la verificación de continuidad temporal que exige el criterio de identificación.

In [6]:
def es_serie_continua(lista_anios):
    lista_anios = sorted(set(lista_anios))  # ordenar y eliminar duplicados
    rango_esperado = list(range(min(lista_anios), max(lista_anios) + 1))
    return lista_anios == rango_esperado

### Episodios por país de referencia

Identifica, para cada uno de los tres países, cuántos de sus productos alcanzaron un episodio
de crecimiento exportador sostenido.

In [7]:
df = pd.DataFrame()

for producto, datos in df_paises.query("ReporterISO3 == 'CRI' and ProductCode not in @list_productos").groupby('ProductCode'):
    df_prueba = datos.sort_values("Year")
    df_prueba.set_index('Year', inplace=True)

    max_year = df_prueba['share'].idxmax()
    if df_prueba[df_prueba.index < max_year].empty == False:
        min_year = df_prueba[df_prueba.index < max_year]['share'].idxmin()

        df_prueba['anio_min'] = min_year
        df_prueba['anio_max'] = max_year
        
        df_prueba.reset_index(inplace=True)

        df_prueba['periodo'] = np.where(df_prueba['Year'] < min_year, 'previo',np.where(df_prueba['Year'] > max_year, 'posterior', 'actual'))
        if es_serie_continua(df_prueba.query("periodo == 'actual'")['Year'].to_list()) == True:
            df_prueba['g%'] = df_prueba['TradeValue in 1000 USD'].pct_change()
            df_prueba['g%_mean'] = df_prueba.groupby('periodo')['g%'].transform('mean')
            df_prueba['Duración'] = max_year - min_year + 1
            df = pd.concat([df, df_prueba], ignore_index=True)

productos_episodio_CRI = df.query("Duración >= 3 and periodo == 'actual' and `g%_mean` >= 0.06")['ProductCode'].nunique()


df = pd.DataFrame()

for producto, datos in df_paises.query("ReporterISO3 == 'MEX' and ProductCode not in @list_productos").groupby('ProductCode'):
    df_prueba = datos.sort_values("Year")
    df_prueba.set_index('Year', inplace=True)

    max_year = df_prueba['share'].idxmax()
    if df_prueba[df_prueba.index < max_year].empty == False:
        min_year = df_prueba[df_prueba.index < max_year]['share'].idxmin()

        df_prueba['anio_min'] = min_year
        df_prueba['anio_max'] = max_year
        
        df_prueba.reset_index(inplace=True)

        df_prueba['periodo'] = np.where(df_prueba['Year'] < min_year, 'previo',np.where(df_prueba['Year'] > max_year, 'posterior', 'actual'))
        if es_serie_continua(df_prueba.query("periodo == 'actual'")['Year'].to_list()) == True:
            df_prueba['g%'] = df_prueba['TradeValue in 1000 USD'].pct_change()
            df_prueba['g%_mean'] = df_prueba.groupby('periodo')['g%'].transform('mean')
            df_prueba['Duración'] = max_year - min_year + 1
            df = pd.concat([df, df_prueba], ignore_index=True)

productos_episodio_MEX = df.query("Duración >= 3 and periodo == 'actual' and `g%_mean` >= 0.06")['ProductCode'].nunique()


df = pd.DataFrame()

for producto, datos in df_paises.query("ReporterISO3 == 'DOM' and ProductCode not in @list_productos").groupby('ProductCode'):
    df_prueba = datos.sort_values("Year")
    df_prueba.set_index('Year', inplace=True)

    max_year = df_prueba['share'].idxmax()
    if df_prueba[df_prueba.index < max_year].empty == False:
        min_year = df_prueba[df_prueba.index < max_year]['share'].idxmin()

        df_prueba['anio_min'] = min_year
        df_prueba['anio_max'] = max_year
        
        df_prueba.reset_index(inplace=True)

        df_prueba['periodo'] = np.where(df_prueba['Year'] < min_year, 'previo',np.where(df_prueba['Year'] > max_year, 'posterior', 'actual'))
        if es_serie_continua(df_prueba.query("periodo == 'actual'")['Year'].to_list()) == True:
            df_prueba['g%'] = df_prueba['TradeValue in 1000 USD'].pct_change()
            df_prueba['g%_mean'] = df_prueba.groupby('periodo')['g%'].transform('mean')
            df_prueba['Duración'] = max_year - min_year + 1
            df = pd.concat([df, df_prueba], ignore_index=True)

productos_episodio_DOM = df.query("Duración >= 3 and periodo == 'actual' and `g%_mean` >= 0.06")['ProductCode'].nunique()

### Proporción de éxito

La proporción resultante es la que pondera las proyecciones del modelo, de modo que los
resultados reflejen tasas de éxito observadas y no la hipótesis de que todos los productos
consolidan su auge.

In [8]:
print(productos_episodio_CRI, productos_episodio_MEX, productos_episodio_DOM)

697 2485 917


In [9]:
print(productos_episodio_CRI/ productos_CRI, productos_episodio_MEX/ productos_MEX, productos_episodio_DOM/ productos_DOM)

0.20433890354734682 0.5443592552026287 0.2656431054461182


In [10]:
np.mean([productos_episodio_CRI/ productos_CRI, productos_episodio_MEX/ productos_MEX, productos_episodio_DOM/ productos_DOM])

0.33811375473203126